# 🔍 Multi-BHR Real-Time Detection

**Detects ALL affected BHR routers simultaneously**
- ⏱️ Sliding window detection (no future data)
- 🗺️ Heatmap showing all detected BHR routers
- 📊 Per-window multi-router detection

In [ ]:
#@title 1️⃣ Setup
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️ Device: {device}")

FEATURE_COLUMNS = [
    'flit_in', 'flit_out', 'avg_wait', 'max_wait', 'buffer_occ', 
    'active_vcs', 'stalls', 'credits', 'crossbar', 'io_ratio',
    'sw_in_arb', 'sw_out_arb', 'empty_vcs', 'total_wait', 
    'min_cred', 'max_cred', 'credit_sends'
]

class BHRAutoencoder(nn.Module):
    def __init__(self, input_dim=17, latent_dim=4):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 12), nn.ReLU(), nn.BatchNorm1d(12),
            nn.Linear(12, 8), nn.ReLU(), nn.BatchNorm1d(8),
            nn.Linear(8, latent_dim), nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 8), nn.ReLU(), nn.BatchNorm1d(8),
            nn.Linear(8, 12), nn.ReLU(), nn.BatchNorm1d(12),
            nn.Linear(12, input_dim)
        )
    def forward(self, x): return self.decoder(self.encoder(x))
    def get_error(self, x):
        with torch.no_grad(): return torch.mean((x - self.forward(x)) ** 2, dim=1)

print("✅ Setup complete")

In [ ]:
#@title 2️⃣ Upload & Train on Normal Data
print("📤 Upload anomaly_features_p0_normal.csv (normal traffic)")
uploaded = files.upload()
df_train = pd.read_csv(list(uploaded.keys())[0]).replace([np.inf, -np.inf], np.nan).dropna()
print(f"✅ Loaded {len(df_train)} samples")

X = df_train[FEATURE_COLUMNS].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

n_val = int(len(X_scaled) * 0.2)
idx = np.random.permutation(len(X_scaled))
X_train = torch.FloatTensor(X_scaled[idx[n_val:]]).to(device)
X_val = torch.FloatTensor(X_scaled[idx[:n_val]]).to(device)

# Train
model = BHRAutoencoder().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

print("🏋️ Training...")
best_loss, best_state = float('inf'), None
for epoch in range(100):
    model.train()
    for i in range(0, len(X_train), 256):
        optimizer.zero_grad()
        loss = criterion(model(X_train[i:i+256]), X_train[i:i+256])
        loss.backward(); optimizer.step()
    model.eval()
    with torch.no_grad(): val_loss = criterion(model(X_val), X_val).item()
    if val_loss < best_loss: best_loss, best_state = val_loss, model.state_dict().copy()
    if (epoch+1) % 25 == 0: print(f"   Epoch {epoch+1}: Val={val_loss:.6f}")

model.load_state_dict(best_state)
errors = model.get_error(X_train).cpu().numpy()
threshold = np.mean(errors) + 3 * np.std(errors)
print(f"\n✅ Trained! Threshold: {threshold:.6f}")

In [ ]:
#@title 3️⃣ Upload Test Data (Attack Traffic)
print("📤 Upload attack data (e.g., anomaly_features_p01_bhr5912.csv)")
uploaded_test = files.upload()
test_filename = list(uploaded_test.keys())[0]
df_test = pd.read_csv(test_filename).replace([np.inf, -np.inf], np.nan).dropna()
print(f"✅ Loaded {len(df_test)} samples from {test_filename}")

unique_ticks = sorted(df_test['tick'].unique())
unique_routers = sorted(df_test['router_id'].unique())
print(f"   Time windows: {len(unique_ticks)}, Routers: {len(unique_routers)}")

In [ ]:
#@title 4️⃣ ⚙️ Set Detection Parameters
#@markdown Anomaly rate threshold for BHR classification
BHR_THRESHOLD = 0.3  #@param {type:"slider", min:0.1, max:0.9, step:0.05}
#@markdown Minimum z-score to mark as anomalous
Z_SCORE_THRESHOLD = 2.0  #@param {type:"slider", min:0.5, max:4.0, step:0.5}

print(f"📊 Detection Parameters:")
print(f"   BHR Threshold: {BHR_THRESHOLD:.0%} anomaly rate")
print(f"   Z-Score Threshold: {Z_SCORE_THRESHOLD}")

In [ ]:
#@title 5️⃣ ⏱️ Sliding Window Multi-BHR Detection
# Process each time window
window_results = []
cumulative_anomalies = {r: 0 for r in unique_routers}
cumulative_samples = {r: 0 for r in unique_routers}

print("🔍 Processing time windows...")
model.eval()

for i, tick in enumerate(unique_ticks):
    window_df = df_test[df_test['tick'] == tick]
    X_window = scaler.transform(window_df[FEATURE_COLUMNS].values)
    scores = model.get_error(torch.FloatTensor(X_window).to(device)).cpu().numpy()
    is_anomaly = scores > threshold
    
    window_router_scores = {}
    for j, (_, row) in enumerate(window_df.iterrows()):
        rid = row['router_id']
        window_router_scores[rid] = scores[j]
        cumulative_anomalies[rid] += int(is_anomaly[j])
        cumulative_samples[rid] += 1
    
    # Calculate rates for all routers
    rates = {r: cumulative_anomalies[r]/max(1,cumulative_samples[r]) for r in unique_routers}
    
    # Identify ALL suspected BHRs (not just top one)
    all_rates = list(rates.values())
    mean_rate = np.mean(all_rates)
    std_rate = np.std(all_rates) + 1e-8
    
    suspected_bhrs = []
    for rid, rate in rates.items():
        z_score = (rate - mean_rate) / std_rate
        if rate > BHR_THRESHOLD and z_score > Z_SCORE_THRESHOLD:
            suspected_bhrs.append({'router': rid, 'rate': rate, 'z_score': z_score})
    
    window_results.append({
        'tick': tick,
        'window_idx': i,
        'suspected_bhrs': suspected_bhrs,
        'router_scores': window_router_scores.copy(),
        'router_rates': rates.copy()
    })
    
    if (i+1) % 1000 == 0:
        bhr_ids = [b['router'] for b in suspected_bhrs]
        print(f"   Window {i+1}/{len(unique_ticks)} - Detected BHRs: {bhr_ids}")

print(f"\n✅ Processed {len(window_results)} windows")

In [ ]:
#@title 6️⃣ 🚨 Final Multi-BHR Detection Results
final = window_results[-1]
final_rates = final['router_rates']

# Calculate z-scores for all routers
all_rates = list(final_rates.values())
mean_rate = np.mean(all_rates)
std_rate = np.std(all_rates) + 1e-8

# Find all BHRs
detected_bhrs = []
for rid in unique_routers:
    rate = final_rates[rid]
    z_score = (rate - mean_rate) / std_rate
    if rate > BHR_THRESHOLD and z_score > Z_SCORE_THRESHOLD:
        detected_bhrs.append({'router': rid, 'rate': rate, 'z_score': z_score})

print("\n" + "="*70)
print("  🚨 MULTI-BHR DETECTION RESULTS")
print("="*70)

if detected_bhrs:
    print(f"\n📍 DETECTED {len(detected_bhrs)} BHR ROUTER(S):")
    for bhr in sorted(detected_bhrs, key=lambda x: x['rate'], reverse=True):
        print(f"   🔴 Router {bhr['router']:2d}: Anomaly Rate = {bhr['rate']:.1%}, Z-Score = {bhr['z_score']:.2f}")
else:
    print("\n✅ No BHR routers detected")

# Show all router stats
print("\n" + "-"*70)
print("  PER-ROUTER SUMMARY")
print("-"*70)
print(f"{'Router':>8} {'Anomaly Rate':>14} {'Z-Score':>10} {'Status':>10}")
print("-"*70)

sorted_routers = sorted(unique_routers, key=lambda r: final_rates[r], reverse=True)
for rid in sorted_routers:
    rate = final_rates[rid]
    z_score = (rate - mean_rate) / std_rate
    status = "🔴 BHR" if any(b['router'] == rid for b in detected_bhrs) else ""
    print(f"{rid:>8} {rate:>13.1%} {z_score:>10.2f} {status:>10}")

In [ ]:
#@title 7️⃣ 🗺️ Multi-BHR Heatmap
# Build heatmap matrix
sample_rate = max(1, len(window_results) // 100)
sampled_windows = window_results[::sample_rate]

heatmap_data = np.zeros((len(unique_routers), len(sampled_windows)))
for j, w in enumerate(sampled_windows):
    for i, rid in enumerate(unique_routers):
        if rid in w['router_scores']:
            heatmap_data[i, j] = w['router_scores'][rid]

# Create heatmap with BHR routers highlighted
fig, ax = plt.subplots(figsize=(16, 8))

# Plot heatmap
sns.heatmap(heatmap_data, 
            yticklabels=unique_routers,
            cmap='YlOrRd',
            ax=ax,
            cbar_kws={'label': 'Anomaly Score'})

# Highlight detected BHR routers with blue boxes
bhr_router_ids = [b['router'] for b in detected_bhrs]
for rid in bhr_router_ids:
    idx = unique_routers.index(rid)
    ax.axhline(y=idx, color='blue', linewidth=2, linestyle='-')
    ax.axhline(y=idx+1, color='blue', linewidth=2, linestyle='-')
    ax.text(-0.5, idx+0.5, '🔴', fontsize=12, va='center')

# X-axis labels (time)
n_labels = 10
step = len(sampled_windows) // n_labels
xticks = list(range(0, len(sampled_windows), step))
xlabels = [f"{sampled_windows[i]['tick']//1000000}M" for i in xticks]
ax.set_xticks(xticks)
ax.set_xticklabels(xlabels)

ax.set_title(f'🗺️ Router × Time Heatmap (Detected BHRs: {bhr_router_ids})', fontsize=14)
ax.set_xlabel('Time (tick)')
ax.set_ylabel('Router ID')

plt.tight_layout()
plt.savefig('multi_bhr_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"📸 Saved: multi_bhr_heatmap.png")

In [ ]:
#@title 8️⃣ 📊 Anomaly Rate Bar Chart
fig, ax = plt.subplots(figsize=(14, 6))

# Colors: red for BHR, blue for normal
colors = ['red' if rid in bhr_router_ids else 'steelblue' for rid in sorted_routers]

bars = ax.bar(range(len(sorted_routers)), 
              [final_rates[r] for r in sorted_routers], 
              color=colors)

ax.set_xticks(range(len(sorted_routers)))
ax.set_xticklabels([f'R{r}' for r in sorted_routers])
ax.set_ylabel('Anomaly Rate')
ax.set_title(f'🎯 Final Anomaly Rate by Router (Detected: {bhr_router_ids})')
ax.axhline(y=BHR_THRESHOLD, color='gray', linestyle='--', label=f'Threshold={BHR_THRESHOLD:.0%}')
ax.legend()

# Add labels on BHR bars
for i, rid in enumerate(sorted_routers):
    if rid in bhr_router_ids:
        ax.annotate(f'{final_rates[rid]:.1%}', 
                   xy=(i, final_rates[rid]), 
                   ha='center', va='bottom', fontsize=10, color='red')

plt.tight_layout()
plt.savefig('multi_bhr_bar.png', dpi=150)
plt.show()

In [ ]:
#@title 9️⃣ ⏱️ BHR Detection Timeline
# Track when each BHR was first detected
first_detection = {rid: None for rid in unique_routers}

for w in window_results:
    for bhr in w['suspected_bhrs']:
        rid = bhr['router']
        if first_detection[rid] is None:
            first_detection[rid] = {'tick': w['tick'], 'window': w['window_idx']}

print("\n" + "="*70)
print("  ⏱️ FIRST DETECTION TIMELINE")
print("="*70)
for rid in bhr_router_ids:
    if first_detection[rid]:
        print(f"   Router {rid}: First detected at tick {first_detection[rid]['tick']} (window {first_detection[rid]['window']})")

# Plot confidence over time for each BHR
if bhr_router_ids:
    fig, ax = plt.subplots(figsize=(14, 6))
    
    for rid in bhr_router_ids:
        rates_over_time = [w['router_rates'].get(rid, 0) for w in window_results[::10]]
        ticks_over_time = [w['tick'] for w in window_results[::10]]
        ax.plot(ticks_over_time, rates_over_time, label=f'Router {rid}', linewidth=2)
    
    ax.axhline(y=BHR_THRESHOLD, color='gray', linestyle='--', label='Threshold')
    ax.set_xlabel('Time (tick)')
    ax.set_ylabel('Cumulative Anomaly Rate')
    ax.set_title('📈 BHR Detection Confidence Over Time')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('multi_bhr_timeline.png', dpi=150)
    plt.show()

In [ ]:
#@title 🔟 💾 Save & Download Results
# Save model
torch.save({'model_state': model.state_dict(), 'scaler': scaler, 'threshold': threshold}, 
           'bhr_autoencoder.pth')

# Save detection summary
summary_data = []
for rid in unique_routers:
    rate = final_rates[rid]
    z_score = (rate - mean_rate) / std_rate
    summary_data.append({
        'router_id': rid,
        'anomaly_rate': rate,
        'z_score': z_score,
        'is_bhr': rid in bhr_router_ids,
        'first_detection_tick': first_detection[rid]['tick'] if first_detection[rid] else None
    })
pd.DataFrame(summary_data).to_csv('detection_summary.csv', index=False)

print("📥 Downloading files...")
files.download('bhr_autoencoder.pth')
files.download('detection_summary.csv')
files.download('multi_bhr_heatmap.png')
files.download('multi_bhr_bar.png')
files.download('multi_bhr_timeline.png')

print("\n✅ Complete! Detected BHR Routers:", bhr_router_ids)